# Notebook 005. Old-growth/non-old-growth forest reference label construction
-------

**WARNING:** SKIP THIS NOTEBOOK IF NOT USING FCC'S ORIGINAL FOREST MANAGEMENT PLANS. The *published dataset* cell of notebook 003 writes this notebook's output directly from the old-growth forest prediction rasters downloaded from Zenodo.

This notebook builds the OGF / non-OGF reference labels from the cleaned forest records of notebook 003:
- Enrich parcels from the management plans and the ownership layer;
- Apply the old-growth / non-old-growth criteria;
- clip roads and disturbance from the old-growth geometries;
- write `ogf_reference_labels.gpkg`

## Enrich from management plans

Fill each parcel's composition and age from the dated record closest to 2020: the 2018 cadastral baseline, or a plan overlapping the parcel at least 90% both ways.

In [ ]:
import geopandas as gpd
import pandas as pd

from utils.overlay import overlap_pairs
from utils.paths import get_project_paths

OVERLAP_THRESHOLD = 0.90  # plan must overlap the parcel this much, both ways, to enrich it
TARGET_YEAR = 2020  # dated record closest to this year wins
BASELINE_YEAR = 2018  # parcel-map cadastral baseline year

paths = get_project_paths()
records_dir = paths.processed / "vectors" / "forest_records"
required = [records_dir / "parcel_map_clean.gpkg", records_dir / "management_plans_clean.gpkg"]
missing = [path.name for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Forest records not available ({', '.join(missing)} missing in {records_dir}). Skip "
        "this notebook: notebook 003 rebuilds the reference labels from the published dataset."
    )
parcels = gpd.read_file(records_dir / "parcel_map_clean.gpkg")
plans = gpd.read_file(records_dir / "management_plans_clean.gpkg")
print(f"[load] {len(parcels)} parcels, {len(plans)} plan polygons")

pairs = overlap_pairs(
    parcels,
    plans,
    source_columns=["species_composition", "average_stand_age", "management_plan_year"],
)
valid = pairs[
    (pairs["overlap_target"] >= OVERLAP_THRESHOLD) & (pairs["overlap_source"] >= OVERLAP_THRESHOLD)
].copy()
print(
    f"[overlap] {len(pairs)} intersecting -> {len(valid)} above {OVERLAP_THRESHOLD:.0%} both ways "
    f"({valid['parcel_id'].nunique()} parcels matched)"
)

# Candidate pool: the 2018 baseline plus every qualifying plan record, all dated.
baseline = pd.DataFrame(
    {
        "parcel_id": parcels["parcel_id"].to_numpy(),
        "year": BASELINE_YEAR,
        "composition": parcels["species_composition"].to_numpy(),
        "age": parcels["average_stand_age"].to_numpy(),
    }
)
alternatives = valid.rename(
    columns={
        "management_plan_year": "year",
        "species_composition": "composition",
        "average_stand_age": "age",
    }
)[["parcel_id", "year", "composition", "age"]]
pool = pd.concat([baseline, alternatives], ignore_index=True)
pool["distance"] = (pool["year"] - TARGET_YEAR).abs()
pool["prefer_after"] = (pool["year"] > TARGET_YEAR).astype(int)  # tiebreak: prefer <= target year


def closest_to_target(attribute):
    ranked = pool[pool[attribute].notna()].sort_values(
        ["parcel_id", "distance", "prefer_after", "year"], ascending=[True, True, True, False]
    )
    return ranked.groupby("parcel_id")[attribute].first()


parcels["final_composition"] = parcels["parcel_id"].map(closest_to_target("composition"))
parcels["final_age"] = parcels["parcel_id"].map(closest_to_target("age")).astype("Int64")

n = len(parcels)
base_comp, final_comp = (
    int(parcels["species_composition"].notna().sum()),
    int(parcels["final_composition"].notna().sum()),
)
base_age, final_age = (
    int(parcels["average_stand_age"].notna().sum()),
    int(parcels["final_age"].notna().sum()),
)
print(
    f"[composition] {base_comp} -> {final_comp} parcels "
    f"({base_comp / n:.0%} -> {final_comp / n:.0%})"
)
print(f"[age] {base_age} -> {final_age} parcels ({base_age / n:.0%} -> {final_age / n:.0%})")

## Enrich ownership

Assign each parcel the ownership type covering at least 90% of its area.

In [ ]:
import geopandas as gpd

from utils.overlay import overlap_pairs
from utils.paths import get_project_paths

OWNERSHIP_THRESHOLD = 0.90  # a parcel takes the ownership type covering at least this share of it

paths = get_project_paths()
owners = gpd.read_file(paths.processed / "vectors" / "forest_records" / "ownership_type_clean.gpkg")

pairs = overlap_pairs(parcels, owners, source_columns=["ownership_type"])
share = (
    pairs.groupby(["parcel_id", "ownership_type"])["overlap_target"]
    .sum()
    .reset_index()
    .sort_values(["parcel_id", "overlap_target"], ascending=[True, False])
)
dominant = share.groupby("parcel_id").first()
dominant = dominant[dominant["overlap_target"] >= OWNERSHIP_THRESHOLD]
parcels["ownership_type"] = parcels["parcel_id"].map(dominant["ownership_type"])

assigned = int(parcels["ownership_type"].notna().sum())
print(
    f"[ownership] {assigned} of {len(parcels)} parcels assigned a dominant type "
    f"(>= {OWNERSHIP_THRESHOLD:.0%})"
)
counts = parcels["ownership_type"].value_counts(dropna=False)
print("[type] " + ", ".join(f"{t}: {n}" for t, n in counts.items()))

## Label old-growth and non-old-growth

Apply the three old-growth criteria (virgin overlap, disturbance record, stand age), and label parcels aged 1 to 80 as non-old-growth.

In [ ]:
import geopandas as gpd
import pandas as pd

from utils.overlay import overlap_pairs
from utils.paths import get_project_paths

OGF_OVERLAP_THRESHOLD = 0.90  # minimum share of a parcel covered by virgin forest to qualify
OGF_MIN_AGE = 80  # parcels younger than this (where age is known) are not old-growth

paths = get_project_paths()
virgin = gpd.read_file(paths.processed / "vectors" / "forest_records" / "virgin_forests_clean.gpkg")

# (i) spatial: share of each parcel covered by virgin/quasi-virgin forest.
pairs = overlap_pairs(parcels, virgin)
overlap = pairs.groupby("parcel_id")["overlap_target"].sum()
parcels["ogf_overlap_ratio"] = parcels["parcel_id"].map(overlap).fillna(0.0)
spatial = parcels["ogf_overlap_ratio"] >= OGF_OVERLAP_THRESHOLD

# (iii) record: drop parcels the cadastral map flags as disturbed (PV anything but "Da").
pv = parcels["padure_virgina_pv"]
pv_ok = pv.isna() | (pv == "Da")

# (ii) age: keep where the recorded age is at least the threshold, or unknown.
age = parcels["final_age"]
age_ok = age.isna() | (age >= OGF_MIN_AGE)

parcels["ogf"] = pd.Series(pd.NA, index=parcels.index, dtype="boolean")
parcels.loc[spatial & pv_ok & age_ok, "ogf"] = True
parcels.loc[parcels["ogf"].isna() & age.notna() & age.between(1, OGF_MIN_AGE), "ogf"] = False

ogf_true = parcels["ogf"].fillna(False).to_numpy(dtype=bool)
ogf_false = (~parcels["ogf"]).fillna(False).to_numpy(dtype=bool)
n_pv_excl = int((spatial & ~pv_ok).sum())
n_age_excl = int((spatial & pv_ok & age.notna() & (age < OGF_MIN_AGE)).sum())
print(
    f"[candidates] {int(spatial.sum())} parcels >= {OGF_OVERLAP_THRESHOLD:.0%} virgin overlap; "
    f"{n_pv_excl} excluded by record, {n_age_excl} by age"
)
print(
    f"[ogf] {int(ogf_true.sum())} old-growth, {int(ogf_false.sum())} non-old-growth, "
    f"{int(parcels['ogf'].isna().sum())} unlabelled"
)
ogf_ha = parcels.loc[ogf_true].geometry.area.sum() / 1e4
non_ha = parcels.loc[ogf_false].geometry.area.sum() / 1e4
print(f"[area] old-growth {ogf_ha:,.0f} ha (pre-clip), " f"non-old-growth {non_ha:,.0f} ha")

## Remove roads and disturbance from old-growth

Trim roads and trails (10 m buffer) and 1985 to 2020 forest disturbance from the old-growth geometries.

In [ ]:
import geopandas as gpd

from utils.geometry_ops import remove_roads_and_disturbance
from utils.paths import get_project_paths
from utils.terminology import OGF_ROAD_BUFFER_M

paths = get_project_paths()
disturbance_path = (
    paths.processed
    / "rasters"
    / "european_forest_disturbance_atlas_10m"
    / "disturbance_1985_2020_3035_10m.tif"
)
roads_path = paths.processed / "vectors" / "open_street_map" / "osm_roads_aoi_buffer_10km.gpkg"

# Roads and trails (buffered OGF_ROAD_BUFFER_M each side) and the 1985-2020 disturbance
# pixels are cut out of the old-growth geometries; parcels left with no geometry are dropped.
# The same helper rebuilds the labels from the published dataset (utils.published).
ogf_idx = parcels["ogf"].fillna(False).to_numpy(dtype=bool)
parcels, cut = remove_roads_and_disturbance(
    parcels,
    ogf_idx,
    roads=gpd.read_file(roads_path),
    disturbance_path=disturbance_path,
    road_buffer_m=OGF_ROAD_BUFFER_M,
)
print(f"[disturbance] {cut.n_patches} disturbed patches within old-growth")
print(
    f"[clip] old-growth {cut.area_before_ha:,.0f} -> {cut.area_after_ha:,.0f} ha "
    f"({cut.area_removed_ha:,.0f} ha removed; {cut.n_removed} parcels removed entirely)"
)

## Classify by species composition

Derive coniferous and broadleaf shares and the dominant species, and assign each parcel a forest-type class.

In [ ]:
import pandas as pd

from utils.forestry import classify_forest_type, composition_metrics
from utils.terminology import SPECIES

metrics = pd.DataFrame(
    [composition_metrics(c) for c in parcels["final_composition"]],
    index=parcels.index,
)
parcels["pct_coniferous"] = metrics["pct_coniferous"]
parcels["pct_broadleaf"] = metrics["pct_broadleaf"]
parcels["dominant_code"] = metrics["dominant_code"]
english = {c: s.english for c, s in SPECIES.items()}
latin = {c: s.latin for c, s in SPECIES.items()}
romanian = {c: s.romanian for c, s in SPECIES.items()}
parcels["dominant_english"] = parcels["dominant_code"].map(english)
parcels["dominant_latin"] = parcels["dominant_code"].map(latin)
parcels["dominant_romanian"] = parcels["dominant_code"].map(romanian)

forest_type = pd.Series(
    [
        classify_forest_type(b, c)
        for b, c in zip(parcels["pct_broadleaf"], parcels["pct_coniferous"], strict=True)
    ],
    index=parcels.index,
)
is_broadleaf = forest_type == "broadleaf"
is_coniferous = forest_type == "coniferous"
is_mixed = forest_type == "mixed"
ogf_true = parcels["ogf"].fillna(False).astype(bool)
ogf_false = (~parcels["ogf"]).fillna(False).astype(bool)

parcels["class"] = pd.NA
parcels.loc[ogf_true & is_broadleaf, "class"] = "OGF_Broadleaf"
parcels.loc[ogf_true & is_coniferous, "class"] = "OGF_Coniferous"
parcels.loc[ogf_true & is_mixed, "class"] = "OGF_Mixed"
parcels.loc[ogf_false & is_broadleaf, "class"] = "Non_OGF_Broadleaf"
parcels.loc[ogf_false & is_coniferous, "class"] = "Non_OGF_Coniferous"
parcels.loc[ogf_false & is_mixed, "class"] = "Non_OGF_Mixed"
parcels.loc[parcels["class"].isna() & ogf_true, "class"] = "OGF_Unclassified"
parcels.loc[parcels["class"].isna() & ogf_false, "class"] = "Non_OGF_Unclassified"
parcels.loc[parcels["class"].isna(), "class"] = "Unlabelled"

n_dominant = int(parcels["dominant_code"].notna().sum())
print(f"[species] dominant species assigned to {n_dominant:,} parcels")
for name, count in parcels["class"].value_counts().sort_index().items():
    print(f"[class] {name}: {count:,}")

## Attach the CORINE forest type

Assign each parcel its dominant CORINE Land Cover forest class by area.

In [ ]:
import geopandas as gpd

from utils.overlay import overlap_pairs
from utils.paths import get_project_paths

paths = get_project_paths()
corine = gpd.read_file(
    paths.processed / "vectors" / "corine_land_cover" / "corine_forest_aoi_3035.gpkg"
)

# Dominant CORINE forest class per parcel: the class covering the greatest parcel share.
pairs = overlap_pairs(parcels, corine, source_columns=["forest_type"])
corine_share = pairs.groupby(["parcel_id", "forest_type"])["overlap_target"].sum().reset_index()
dominant = corine_share.sort_values("overlap_target", ascending=False).groupby("parcel_id").first()
parcels["corine_forest_type"] = parcels["parcel_id"].map(dominant["forest_type"])

assigned = int(parcels["corine_forest_type"].notna().sum())
print(f"[corine] forest type assigned to {assigned:,} of {len(parcels):,} parcels")
counts = parcels["corine_forest_type"].value_counts(dropna=False)
print("[corine] " + ", ".join(f"{t}: {n}" for t, n in counts.items()))

## Write the reference labels

Repair the geometries and write the labelled parcel layer, then report per-class counts and areas.

In [ ]:
import logging

import geopandas as gpd

from utils.paths import get_project_paths
from utils.vector_io import repair_geometries

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
output_path = paths.labels / "ogf_reference_labels.gpkg"

parcels = repair_geometries(parcels)

output_columns = [
    "parcel_id",
    "ogf",
    "class",
    "final_composition",
    "final_age",
    "pct_coniferous",
    "pct_broadleaf",
    "dominant_code",
    "dominant_english",
    "dominant_latin",
    "dominant_romanian",
    "ogf_overlap_ratio",
    "ownership_type",
    "corine_forest_type",
    "geometry",
]
labels = parcels[output_columns].copy()
output_path.parent.mkdir(parents=True, exist_ok=True)
labels.to_file(output_path, layer="parcels", driver="GPKG")
print(f"[write] {len(labels):,} parcels -> {output_path}")

readback = gpd.read_file(output_path, layer="parcels")
invalid = int((~readback.geometry.is_valid).sum())
print(f"[verify] {len(readback):,} rows read back, {invalid} invalid geometries")
summary = (
    readback.assign(area_ha=readback.geometry.area / 1e4)
    .groupby("class")
    .agg(n=("class", "size"), area_ha=("area_ha", "sum"))
    .sort_index()
)
print(summary.to_string(float_format=lambda v: f"{v:,.0f}"))

## Threshold sensitivity

Recompute the old-growth and non-old-growth counts as the overlap thresholds vary from 0.70 to 0.99, then plot the count sensitivity, with the 0.90 threshold marked.

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

from utils.overlay import overlap_pairs
from utils.paths import get_project_paths

TARGET_YEAR = 2020
BASELINE_YEAR = 2018
OGF_MIN_AGE = 80
THRESHOLDS = np.round(np.arange(0.70, 0.99, 0.01), 2)

paths = get_project_paths()
records = paths.processed / "vectors" / "forest_records"
parcels = gpd.read_file(records / "parcel_map_clean.gpkg")
plans = gpd.read_file(records / "management_plans_clean.gpkg")
virgin = gpd.read_file(records / "virgin_forests_clean.gpkg")

# Overlap pairs do not depend on the threshold, so compute them once.
plan_pairs = overlap_pairs(
    parcels, plans, source_columns=["average_stand_age", "management_plan_year"]
)
virgin_ratio = overlap_pairs(parcels, virgin).groupby("parcel_id")["overlap_target"].sum()
parcels["virgin_ratio"] = parcels["parcel_id"].map(virgin_ratio).fillna(0.0)

baseline = parcels[["parcel_id", "average_stand_age"]].rename(columns={"average_stand_age": "age"})
baseline["year"] = BASELINE_YEAR
pv_ok = parcels["padure_virgina_pv"].isna() | (parcels["padure_virgina_pv"] == "Da")


def closest_age(pool: pd.DataFrame) -> pd.Series:
    """Pick each parcel's age from the source closest to the target year."""
    pool = pool.copy()
    pool["dist"] = (pool["year"] - TARGET_YEAR).abs()
    pool["prefer_after"] = (pool["year"] > TARGET_YEAR).astype(int)
    pool = pool[pool["age"].notna()].sort_values(
        ["parcel_id", "dist", "prefer_after", "year"], ascending=[True, True, True, False]
    )
    return pool.groupby("parcel_id")["age"].first()


rows = []
for t in THRESHOLDS:
    valid = plan_pairs[(plan_pairs["overlap_target"] >= t) & (plan_pairs["overlap_source"] >= t)]
    alt = valid[["parcel_id", "management_plan_year", "average_stand_age"]].rename(
        columns={"management_plan_year": "year", "average_stand_age": "age"}
    )
    pool = pd.concat([baseline, alt], ignore_index=True)
    age = parcels["parcel_id"].map(closest_age(pool))
    spatial = parcels["virgin_ratio"] >= t
    age_ok = age.isna() | (age >= OGF_MIN_AGE)
    is_ogf = spatial & pv_ok & age_ok
    is_non = ~is_ogf & age.between(1, OGF_MIN_AGE)
    rows.append((t, int(is_ogf.sum()), int(is_non.sum())))

sweep = pd.DataFrame(rows, columns=["threshold", "old_growth", "non_old_growth"])
print(sweep.to_string(index=False))
og, non = sweep["old_growth"], sweep["non_old_growth"]
print(
    f"\n[sensitivity] thresholds 0.70-0.99: old-growth {og.min()}-{og.max()}, "
    f"non-old-growth {non.min()}-{non.max()}"
)
base = sweep[sweep["threshold"] == 0.90].iloc[0]
print(f"[baseline] 0.90: {base['old_growth']} old-growth, {base['non_old_growth']} non-old-growth")

import matplotlib.pyplot as plt

from utils.style import CATEGORICAL_PALETTE, get_figure_size, save_figure, use_publication_style

use_publication_style()  # Charis SIL publication typeface (utils.style)

NOTEBOOK = "005_reference_label_construction"

base = sweep[sweep["threshold"] == 0.90].iloc[0]
panels = [
    ("old_growth", "(a) Old-growth", CATEGORICAL_PALETTE[0]),
    ("non_old_growth", "(b) Non-old-growth", CATEGORICAL_PALETTE[4]),
]

fig, axes = plt.subplots(1, 2, figsize=get_figure_size("double", aspect=0.42))
for ax, (column, title, colour) in zip(axes, panels, strict=True):
    ax.plot(sweep["threshold"], sweep[column], color=colour)
    value = base[column]
    ax.axvline(0.90, color="0.5", linestyle="--", linewidth=0.8)
    ax.plot(0.90, value, marker="o", markersize=4, color=colour)
    ax.annotate(
        f"{int(value):,}",
        xy=(0.90, value),
        xytext=(4, 0),
        textcoords="offset points",
        fontsize=8,
        va="center",
    )
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel("Overlap threshold")
    ax.set_ylabel("Parcel count")
fig.tight_layout()
written = save_figure(fig, f"{NOTEBOOK}/threshold_sensitivity", data=sweep)
plt.show()
print(f"[figure] saved {written[0]}")